In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from processing import *
from wavelengths import *
from reprojection import View
from datetime import datetime

In [2]:
q_V = 299792458 / 6173.341
q_B = q_V * 0.231
tuning_constant = 3.513e-4
temperature_constant = 4.01225e-2
alpha = 1.327124e20

In [31]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/vlos_/*.fits'))

In [32]:
dates = []
offsets = []
biases = []
temperatures = []
velocities = []
accelerations = []
distances = []
contposes = []
soops = []

for file in files[:]:

    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header
        fg_data = hdul['PHI_FITS_FG_settings'].data
        pmp_data = hdul['PHI_FITS_PMP_settings'].data


    nx, ny = header['NAXIS2'], header['NAXIS1']
    xc, yc = header['CRPIX2'] - 1, header['CRPIX1'] - 1
    rsun = header['RSUN_ARC'] / header['CDELT1']

    #xi, yi = np.mgrid[:nx, :ny]
    #mask = (xi - xc) ** 2 + (yi - yc) ** 2 < (rsun * 0.2) ** 2
    soop = header['SOOPNAME']
    velocity = header['OBS_VR']
    distance = header['DSUN_AU']
    temperature = header['FGOV1PT1']
    contposn = header['CONTPOSN']
    contpos = header['CONTPOS'] - 1
    date = datetime.fromisoformat(header['DATE-OBS'])
    acceleration = (header['OBS_VW'] ** 2 + header['OBS_VN'] ** 2) / header['DSUN_OBS'] - alpha / header['DSUN_OBS'] ** 2

    wvs = read_wavelengths(header)
    wv0 = np.delete(wvs, contpos)[2]

    data = (data + velocity) / q_V
    biases += [np.nanmedian(data) + 6173.341 - wv0]

    data -= temperature_constant * (temperature - 61)
    data /= tuning_constant ## in Volts

    dates += [date]
    soops += [soop]
    offsets += [np.nanmedian(data)]
    temperatures += [temperature]
    velocities += [velocity / 1000]
    distances += [distance]
    accelerations += [acceleration]
    contposes += [contposn]


dates = np.array(dates)
soops = np.array(soops)
biases = np.array(biases)
offsets = np.array(offsets)
temperatures = np.array(temperatures)
velocities = np.array(velocities)
distances = np.array(distances)
accelerations = np.array(accelerations)
contposes = np.array(contposes)

In [33]:
fig, ax = plt.subplots(figsize=(12,8))
ax1 = ax.twinx()

colors = ['tab:blue', 'tab:green', 'tab:red']
for temperature, color in zip([56, 61, 66], colors):
    t = np.where(np.abs(temperatures - temperature) < 2)
    ax.plot(dates[t], biases[t], '.', color=color, label=temperature)

#ax1.plot(dates, distances, '--', color='gray')
ax1.plot(dates, velocities, '--', color='gray')
#ax1.plot(dates, accelerations, '--', color='gray')

contpos_ = contposes[0]
date_ = dates[0]
for contpos, date in zip(contposes[1:], dates[1:]):
    if contpos != contpos_:
        ax.axvspan(date_, date, color='tab:' + contpos_, alpha=0.1)
        date_ = date
        contpos_ = contpos
ax.axvspan(date_, date, color='tab:' + contpos_, alpha=0.1)


ax.set_xlabel('Date')
ax.set_ylabel(r'Offset, $\AA$')
#ax1.set_ylabel('Distance, AU')
ax1.set_ylabel('Velocity, km/s')
#ax1.set_ylabel(r'Acceleration, m/s$^2$')

ax.set_xlim(dates[0], dates[-1])
#ax.set_ylim(-0.035, 0)

plt.grid(True)
ax.legend()
plt.tight_layout()

In [34]:
def make_plot(offsets, velocities, temperatures, fig=None, ax=None, **kwargs):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10,8))

    ax.scatter(offsets, velocities, **kwargs)


    t_56 = np.where(np.all([offsets < 100,
                            np.abs(temperatures - 56) < 2], axis=0))[0]
    if len(t_56) > 1:
        k_56, b_56 = np.polyfit(offsets[t_56], velocities[t_56], 1)
        plt.plot([850,-850], [k_56 * 850 + b_56, k_56 * -850 + b_56], '--', color='k', lw=0.5)
        print(k_56 / q_V * 1e3, b_56 / q_V * 1e3)


    t_61 = np.where(np.all([offsets > 50,
                            np.abs(temperatures - 61) < 2], axis=0))[0]
    if len(t_61) > 1:
        k_61, b_61 = np.polyfit(offsets[t_61], velocities[t_61], 1)
        plt.plot([-850,850], [k_61 * -850 + b_61, k_61 * 850 + b_61], '--', color='k', lw=0.5)
        print(k_61 / q_V * 1e3, b_61 / q_V * 1e3)


    t_66 = np.where(np.all([offsets < 400,
                            np.abs(temperatures - 66) < 2], axis=0))[0]
    if len(t_66) > 1:
        k_66, b_66 = np.polyfit(offsets[t_66], velocities[t_66], 1)
        plt.plot([-850,850], [k_66 * -850 + b_66, k_66 * 850 + b_66], '--', color='k', lw=0.5)
        print(k_66 / q_V * 1e3, b_66 / q_V * 1e3)


    #print(k_56 / q_V * 1e3, k_61 / q_V * 1e3, k_66 / q_V * 1e3)

    ax.set_xlabel('Offset, V')
    ax.set_ylabel('S/C velocity, km/s')
    ax.legend()

    ax.set_xlim(-1000,1000)
    ax.set_ylim(-30,30)
    ax.grid(True)
    fig.tight_layout()

    return fig, ax

In [35]:
synoptics = np.any([soops == 'R_FULL_LRES_LCAD_RS-Synoptics-Low',
                    soops == 'R_FULL_LRES_LCAD_RS-Synoptics-High',
                    ], axis=0)

t0 = np.where(np.all([~synoptics,
                     dates > datetime(2025,1,1),
                     dates < datetime(2027,1,1)], axis=0))[0]

t1 = np.where(np.all([synoptics,
                     dates > datetime(2025,1,1),
                     dates < datetime(2026,1,1)], axis=0))[0]#[::4]

t2 = np.where(np.all([synoptics,
                     dates > datetime(2026,1,1),
                     dates < datetime(2027,1,1)], axis=0))[0]#[::4]


#fig, ax = make_plot(offsets[t0], velocities[t0], temperatures[t0], label='non-synoptic', marker='.', color='black', s=5)
#fig, ax = make_plot(offsets[t1], velocities[t1], temperatures[t1], label='2025', marker='o', lw=0.5, s=30, fc='none', ec='blue')
fig, ax = make_plot(offsets[t2], velocities[t2], temperatures[t2], label='2026', marker='x', lw=0.5, s=30, color='red')

0.0003344875083114116 0.025948487065415117


In [62]:
(0.1840 + 0.0234) / 5, (0.2203 - 0.0234) / 5

(0.04148, 0.03938)

In [67]:
3.5e-4 * 100

0.034999999999999996